In [ ]:
ENGAGEMENT_WEIGHTS = {
    "information": {
        # Intent weights (base score)
        "Inquiry": 1.0,        # Questions = high engagement (seeking clarity)
        "Appreciation": 0.8,   # Thanks/acknowledgment
        "Statement": 0.6,      # Neutral discussion
        "Criticism": 0.3,      # Concerns (not ideal but engagement)
        "Forward": 0.4,        # Sharing (moderate value for info posts)
        
        # Sentiment multipliers
        "sentiment_multipliers": {
            "Positive": 1.2,
            "Neutral": 1.0,
            "Negative": 0.7,   # Negative reduces score
        }
    },
    
    "branding": {
        # Intent weights (base score)
        "Appreciation": 1.0,   # Praise = highest value
        "Forward": 0.9,        # Sharing = amplification
        "Statement": 0.6,      # Discussion (neutral)
        "Inquiry": 0.5,        # Questions (less ideal for branding)
        "Criticism": 0.2,      # Criticism hurts brand
        
        # Sentiment multipliers
        "sentiment_multipliers": {
            "Positive": 1.3,   # Boost positive more for branding
            "Neutral": 0.9,
            "Negative": 0.5,   # Penalize negative heavily
        }
    }
}

def calculate_engagement_score(comments, post_type):
    """
    Calculate engagement score for a set of comments.
    
    Returns:
    - overall_score: 0-100
    - breakdown: dict with details
    """
    weights = ENGAGEMENT_WEIGHTS[post_type]
    total_score = 0
    intent_counts = {"Appreciation": 0, "Criticism": 0, "Statement": 0, 
                     "Inquiry": 0, "Forward": 0}
    sentiment_counts = {"Positive": 0, "Neutral": 0, "Negative": 0}
    
    for comment in comments:
        intent = comment['intent']
        sentiment = comment['sentiment']
        
        # Base score from intent
        base_score = weights.get(intent, 0.5)
        
        # Multiply by sentiment
        sentiment_mult = weights['sentiment_multipliers'].get(sentiment, 1.0)
        
        # Final comment score
        comment_score = base_score * sentiment_mult
        total_score += comment_score
        
        # Track counts
        intent_counts[intent] += 1
        sentiment_counts[sentiment] += 1
    
    # Normalize to 0-100 scale
    if len(comments) > 0:
        avg_score = total_score / len(comments)
        normalized_score = min(100, avg_score * 100)
    else:
        normalized_score = 0
    
    # Breakdown
    breakdown = {
        "overall_score": round(normalized_score, 2),
        "total_comments": len(comments),
        "intent_distribution": intent_counts,
        "sentiment_distribution": sentiment_counts,
        "post_type": post_type,
        "interpretation": interpret_score(normalized_score, post_type)
    }
    
    return breakdown


def interpret_score(score, post_type):
    """Provide interpretation of the score."""
    if post_type == "information":
        if score >= 80:
            return "Excellent: High engagement with constructive questions and appreciation"
        elif score >= 60:
            return "Good: Moderate engagement with some concerns raised"
        elif score >= 40:
            return "Fair: Mixed engagement with notable criticism"
        else:
            return "Poor: Low engagement or predominantly negative/critical responses"
    else:  # branding
        if score >= 80:
            return "Excellent: Strong positive sentiment and high shareability"
        elif score >= 60:
            return "Good: Positive reception with some neutral discussion"
        elif score >= 40:
            return "Fair: Mixed reception with some criticism"
        else:
            return "Poor: Negative brand perception or low engagement"